In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

nguyenng11_spacenewdata_path = kagglehub.dataset_download('nguyenng11/spacenewdata')
nguyenng11_orgdataset_path = kagglehub.dataset_download('nguyenng11/orgdataset')

print('Data source import complete.')


Data source import complete.


## Setting up...

In [3]:
import os

# 1. Cài đặt phiên bản mới nhất
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate


In [4]:
import bitsandbytes
print(f"Phiên bản bitsandbytes hiện tại: {bitsandbytes.__version__}")
# Kết quả phải từ 0.46.1 trở lên mới đạt yêu cầu của Gemma 3.

Phiên bản bitsandbytes hiện tại: 0.49.2


In [5]:
import os
import glob
import json
import torch
import time
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
import kagglehub

# Lấy Hugging Face token
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HFEval")
    print("✅ Đã tìm thấy HF_TOKEN.")
except:
    print("❌ Lỗi: Bạn chưa tạo Secret 'HF_TOKEN' trên Kaggle.")

✅ Đã tìm thấy HF_TOKEN.


In [6]:
print("Đang nạp mô hình Gemma 3 12B... Vui lòng đợi trong giây lát.")

# Cấu hình nén 4-bit chuẩn NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Sử dụng bản 12B để đảm bảo tốc độ và sự ổn định trên Kaggle
# model_id = "google/gemma-3-12b-it"
model_path = kagglehub.model_download("google/gemma-3/Transformers/gemma-3-12b-it/1")


tokenizer = AutoTokenizer.from_pretrained(model_path, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("🚀 Mô hình đã sẵn sàng trên GPU!")

Đang nạp mô hình Gemma 3 12B... Vui lòng đợi trong giây lát.


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

🚀 Mô hình đã sẵn sàng trên GPU!


## Common Helper Functions

In [7]:
import re
import gc
import os as _os
_os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Các thẻ entity (BỎ QUA MEASURE)
ENTITY_LABELS = ['PLACE', 'PATH', 'SPATIAL_ENTITY', 'MOTION', 'NONMOTION_EVENT', 'SPATIAL_SIGNAL', 'MOTION_SIGNAL']
# Các thẻ relation
LINK_LABELS = ['QSLINK', 'OLINK', 'MOVELINK']
# Các thẻ bỏ qua hoàn toàn
SKIP_LABELS = ['MEASURE', 'MEASURELINK', 'METALINK', 'CP', 'URL']

# Giới hạn text input để tránh OOM
MAX_TEXT_LENGTH = 1500

def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

def extract_json_only(text):
    """Chỉ lấy phần nằm trong dấu { }"""
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}
    except:
        return {}

def ask_gemma_raw(prompt_text):
    """Few-shot: Gửi prompt có kèm ví dụ minh hoạ"""
    clean_memory()  # Dọn VRAM trước mỗi lần gọi
    messages = [{"role": "user", "content": prompt_text}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_formatted, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1200, temperature=0.1)

    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    del inputs, outputs  # Giải phóng VRAM ngay
    clean_memory()
    return result

## Build Few-Shot Examples từ Training Data
Load 2 file training nhỏ nhất để làm ví dụ minh hoạ cho prompt. Mỗi ví dụ gồm TEXT đầu vào và JSON đầu ra mong muốn.

In [8]:
# Đường dẫn thư mục training data
TRAIN_DIR = "/kaggle/input/datasets/nguyenng11/spacenewdata/newdata/train"

def build_fewshot_example(xml_path, max_text_len=500):
    """Đọc 1 file training XML và trích xuất ví dụ few-shot.
    Trả về dict gồm: text, entities (list), relations (list)"""
    with open(xml_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'xml')

    raw_text = soup.find('TEXT').get_text().strip()[:max_text_len]
    tags = soup.find('TAGS')
    if not tags:
        return None

    all_tags = tags.find_all(recursive=False)

    # Parse entities
    entities = []
    for tag in all_tags:
        label = tag.name.upper()
        if label not in ENTITY_LABELS:
            continue
        if tag.get('start') == '-1':
            continue

        ent = {
            'id': tag.get('id', ''),
            'text': tag.get('text', ''),
            'label': label
        }

        # Lấy attributes theo loại entity
        attrs = {}
        for attr, val in tag.attrs.items():
            if attr in ['id', 'start', 'end', 'text', 'comment', 'domain',
                        'continent', 'state', 'country', 'gazref', 'latLong',
                        'elevation', 'mod', 'gquant', 'scopes', 'type',
                        'beginID', 'endID', 'midIDs', 'cluster']:
                continue
            if val:  # Chỉ lấy thuộc tính có giá trị
                attrs[attr] = val
        ent['attributes'] = attrs
        entities.append(ent)

    # Parse relations
    relations = []
    for tag in all_tags:
        label = tag.name.upper()
        if label not in LINK_LABELS:
            continue

        rel = {'type': label.replace('LINK', 'Link')}
        if label == 'QSLINK':
            rel.update({
                'trajector': tag.get('trajector', ''),
                'landmark': tag.get('landmark', ''),
                'trigger': tag.get('trigger', ''),
                'relType': tag.get('relType', '')
            })
        elif label == 'OLINK':
            rel.update({
                'trajector': tag.get('trajector', ''),
                'landmark': tag.get('landmark', ''),
                'trigger': tag.get('trigger', ''),
                'relType': tag.get('relType', ''),
                'frame_type': tag.get('frame_type', ''),
                'referencePt': tag.get('referencePt', ''),
                'projective': tag.get('projective', '')
            })
        elif label == 'MOVELINK':
            rel.update({
                'trigger': tag.get('trigger', ''),
                'mover': tag.get('mover', ''),
                'source': tag.get('source', ''),
                'goal': tag.get('goal', ''),
                'midPoint': tag.get('midPoint', ''),
                'landmark': tag.get('landmark', ''),
                'pathID': tag.get('pathID', ''),
                'motion_signalID': tag.get('motion_signalID', ''),
                'goal_reached': tag.get('goal_reached', '')
            })
        # Fix type name
        if label == 'QSLINK': rel['type'] = 'QSLink'
        elif label == 'OLINK': rel['type'] = 'OLink'
        elif label == 'MOVELINK': rel['type'] = 'MoveLink'

        relations.append(rel)

    return {
        'text': raw_text,
        'entities': entities,
        'relations': relations
    }


def select_fewshot_files(train_dir, n=2, max_size_bytes=15000):
    """Chọn n file training nhỏ nhất để làm ví dụ few-shot (tránh OOM)"""
    all_files = glob.glob(os.path.join(train_dir, '**', '*.xml'), recursive=True)
    # Sắp xếp theo kích thước file, lấy n file nhỏ nhất
    sized = [(f, os.path.getsize(f)) for f in all_files]
    sized.sort(key=lambda x: x[1])
    selected = [f for f, s in sized if s <= max_size_bytes][:n]
    return selected


# Chọn và build few-shot examples
fewshot_files = select_fewshot_files(TRAIN_DIR, n=2)
FEWSHOT_EXAMPLES = []
for fpath in fewshot_files:
    ex = build_fewshot_example(fpath)
    if ex:
        FEWSHOT_EXAMPLES.append(ex)
        print(f"✅ Loaded few-shot example: {os.path.basename(fpath)} ({len(ex['entities'])} entities, {len(ex['relations'])} relations)")

print(f"\n📚 Tổng cộng {len(FEWSHOT_EXAMPLES)} ví dụ few-shot.")

✅ Loaded few-shot example: 47_N_26_E.xml (17 entities, 6 relations)
✅ Loaded few-shot example: 46_N_25_E.xml (27 entities, 7 relations)

📚 Tổng cộng 2 ví dụ few-shot.


In [9]:
def format_entity_examples(examples):
    """Tạo chuỗi few-shot examples cho prompt entities"""
    parts = []
    for i, ex in enumerate(examples):
        ents_json = json.dumps({"entities": ex['entities']}, ensure_ascii=False, indent=2)
        parts.append(f"--- Example {i+1} ---\nTEXT: {ex['text']}\nOUTPUT:\n{ents_json}")
    return "\n\n".join(parts)

def format_relation_examples(examples):
    """Tạo chuỗi few-shot examples cho prompt relations"""
    parts = []
    for i, ex in enumerate(examples):
        ents_brief = json.dumps(ex['entities'], ensure_ascii=False)
        rels_json = json.dumps({"relations": ex['relations']}, ensure_ascii=False, indent=2)
        parts.append(f"--- Example {i+1} ---\nTEXT: {ex['text']}\nENTITIES: {ents_brief}\nOUTPUT:\n{rels_json}")
    return "\n\n".join(parts)

def format_attribute_examples(examples):
    """Tạo chuỗi few-shot examples cho prompt attributes (Config 2)"""
    parts = []
    for i, ex in enumerate(examples):
        # Input: entities CHỈ có id, text, label (không có attributes)
        ents_input = [{"id": e["id"], "text": e["text"], "label": e["label"]} for e in ex['entities']]
        # Output: entities VỚI attributes
        ents_output = [{"id": e["id"], "attributes": e["attributes"]} for e in ex['entities']]
        ents_in_json = json.dumps(ents_input, ensure_ascii=False)
        ents_out_json = json.dumps({"entities": ents_output}, ensure_ascii=False, indent=2)
        parts.append(f"--- Example {i+1} ---\nTEXT: {ex['text']}\nENTITIES: {ents_in_json}\nOUTPUT:\n{ents_out_json}")
    return "\n\n".join(parts)

# Pre-build các chuỗi examples
ENTITY_EXAMPLES_STR = format_entity_examples(FEWSHOT_EXAMPLES)
RELATION_EXAMPLES_STR = format_relation_examples(FEWSHOT_EXAMPLES)
ATTRIBUTE_EXAMPLES_STR = format_attribute_examples(FEWSHOT_EXAMPLES)

print("✅ Đã tạo xong chuỗi few-shot examples cho entities, relations, và attributes.")

✅ Đã tạo xong chuỗi few-shot examples cho entities, relations, và attributes.


## Load Gold Standard (chung cho 3 config)

In [10]:
# ĐƯỜNG DẪN ĐẾN THƯ MỤC CHỨA FILE GOLD STANDARD
GOLD_DIR = "/kaggle/input/datasets/canhtungdz/standargold/newdata/gold"

def load_gold_standard(directory):
    gold_dict = {}
    files = glob.glob(os.path.join(directory, '**', '*.xml'), recursive=True)

    for path in files:
        fname = os.path.basename(path)
        with open(path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'xml')
            tags = soup.find('TAGS')
            if tags:
                gold_dict[fname] = tags
    return gold_dict

# Nạp đáp án vào RAM
GOLD_DICT = load_gold_standard(GOLD_DIR)
print(f"✅ Đã nạp đáp án chuẩn cho {len(GOLD_DICT)} file.")

✅ Đã nạp đáp án chuẩn cho 16 file.


## Evaluation Functions

In [11]:
def evaluate_config1(predictions_path, gold_dict):
    """Đánh giá Config 1: Task a+b, c, d, e"""
    results = {
        "1a_1b": {"tp": 0, "fp": 0, "fn": 0},
        "1c": {"correct": 0, "total": 0},
        "1d": {"tp": 0, "fp": 0, "fn": 0},
        "1e": {"correct": 0, "total": 0}
    }

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            filename = record['metadata']['file']
            pred_data = record['data']
            if filename not in gold_dict: continue

            gold_tags = gold_dict[filename]
            all_gold = gold_tags.find_all(recursive=False)
            # Lấy entities gold (bỏ qua MEASURE, links, start=-1)
            gold_ents = [t for t in all_gold if t.name.upper() in ENTITY_LABELS and t.get('start') != "-1"]
            gold_links = [t for t in all_gold if t.name.upper() in LINK_LABELS]

            # --- CHẤM BÀI a+b, c (ENTITIES) ---
            pred_ents = pred_data.get('entities', [])
            matched_gold_ids = {}

            for p in pred_ents:
                p_text = str(p.get('text', '')).strip().lower()
                p_label = str(p.get('label', '')).strip().upper()

                match = None
                for g in gold_ents:
                    g_id = g.get('id', '')
                    if g_id in matched_gold_ids.values(): continue
                    if p_text == str(g.get('text', '')).strip().lower() and p_label == g.name.upper():
                        match = g
                        break

                if match:
                    results["1a_1b"]["tp"] += 1
                    matched_gold_ids[p.get('id')] = match.get('id')

                    # Bài c: Attributes
                    p_attrs = p.get('attributes', {})
                    for attr, g_val in match.attrs.items():
                        if attr in ['id', 'start', 'end', 'text', 'comment']: continue
                        results["1c"]["total"] += 1
                        if str(p_attrs.get(attr, '')).upper() == str(g_val).upper():
                            results["1c"]["correct"] += 1
                else:
                    results["1a_1b"]["fp"] += 1

            results["1a_1b"]["fn"] += (len(gold_ents) - len(matched_gold_ids))

            # --- CHẤM BÀI d, e (RELATIONS) ---
            pred_rels = pred_data.get('relations', [])
            matched_link_count = 0

            for pr in pred_rels:
                p_type = pr.get('type', '').upper()
                is_link_tp = False

                for gr in gold_links:
                    if p_type == gr.name.upper():
                        results["1d"]["tp"] += 1
                        is_link_tp = True
                        matched_link_count += 1

                        # Bài e: Link Attributes
                        for g_attr, g_val in gr.attrs.items():
                            if g_attr in ['id', 'fromID', 'toID', 'fromText', 'toText', 'comment']: continue
                            results["1e"]["total"] += 1
                            if str(pr.get(g_attr, '')).upper() == str(g_val).upper():
                                results["1e"]["correct"] += 1
                        break

                if not is_link_tp:
                    results["1d"]["fp"] += 1

            results["1d"]["fn"] += (len(gold_links) - matched_link_count)

    return results


def evaluate_config2(predictions_path, gold_dict):
    """Đánh giá Config 2: Task c, d, e (entities đã cho sẵn)"""
    results = {
        "1c": {"correct": 0, "total": 0},
        "1d": {"tp": 0, "fp": 0, "fn": 0},
        "1e": {"correct": 0, "total": 0}
    }

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            filename = record['metadata']['file']
            pred_data = record['data']
            if filename not in gold_dict: continue

            gold_tags = gold_dict[filename]
            all_gold = gold_tags.find_all(recursive=False)
            gold_ents = [t for t in all_gold if t.name.upper() in ENTITY_LABELS and t.get('start') != "-1"]
            gold_links = [t for t in all_gold if t.name.upper() in LINK_LABELS]

            # --- Task c: Entity Attributes ---
            # So khớp entities theo id (vì C2 đã cho sẵn entities với id)
            pred_attrs_map = {e.get('id'): e.get('attributes', {}) for e in pred_data.get('entities', [])}
            for g in gold_ents:
                g_id = g.get('id', '')
                if g_id in pred_attrs_map:
                    p_attrs = pred_attrs_map[g_id]
                    for attr, g_val in g.attrs.items():
                        if attr in ['id', 'start', 'end', 'text', 'comment']: continue
                        results["1c"]["total"] += 1
                        if str(p_attrs.get(attr, '')).upper() == str(g_val).upper():
                            results["1c"]["correct"] += 1

            # --- Task d, e: Relations ---
            pred_rels = pred_data.get('relations', [])
            matched_link_count = 0

            for pr in pred_rels:
                p_type = pr.get('type', '').upper()
                is_link_tp = False

                for gr in gold_links:
                    if p_type == gr.name.upper():
                        results["1d"]["tp"] += 1
                        is_link_tp = True
                        matched_link_count += 1

                        for g_attr, g_val in gr.attrs.items():
                            if g_attr in ['id', 'fromID', 'toID', 'fromText', 'toText', 'comment']: continue
                            results["1e"]["total"] += 1
                            if str(pr.get(g_attr, '')).upper() == str(g_val).upper():
                                results["1e"]["correct"] += 1
                        break

                if not is_link_tp:
                    results["1d"]["fp"] += 1

            results["1d"]["fn"] += (len(gold_links) - matched_link_count)

    return results


def evaluate_config3(predictions_path, gold_dict):
    """Đánh giá Config 3: Task d, e (entities + attributes đã cho sẵn)"""
    results = {
        "1d": {"tp": 0, "fp": 0, "fn": 0},
        "1e": {"correct": 0, "total": 0}
    }

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            filename = record['metadata']['file']
            pred_data = record['data']
            if filename not in gold_dict: continue

            gold_tags = gold_dict[filename]
            all_gold = gold_tags.find_all(recursive=False)
            gold_links = [t for t in all_gold if t.name.upper() in LINK_LABELS]

            pred_rels = pred_data.get('relations', [])
            matched_link_count = 0

            for pr in pred_rels:
                p_type = pr.get('type', '').upper()
                is_link_tp = False

                for gr in gold_links:
                    if p_type == gr.name.upper():
                        results["1d"]["tp"] += 1
                        is_link_tp = True
                        matched_link_count += 1

                        for g_attr, g_val in gr.attrs.items():
                            if g_attr in ['id', 'fromID', 'toID', 'fromText', 'toText', 'comment']: continue
                            results["1e"]["total"] += 1
                            if str(pr.get(g_attr, '')).upper() == str(g_val).upper():
                                results["1e"]["correct"] += 1
                        break

                if not is_link_tp:
                    results["1d"]["fp"] += 1

            results["1d"]["fn"] += (len(gold_links) - matched_link_count)

    return results


def display_report(res, config_name):
    print(f"\n===== KẾT QUẢ ĐÁNH GIÁ {config_name} (FEW-SHOT) =====")
    print(f"{'BÀI TOÁN':<25} | {'P':<8} | {'R':<8} | {'F1':<8} | {'ACC':<8}")
    print("-" * 70)

    if "1a_1b" in res:
        tp, fp, fn = res["1a_1b"]["tp"], res["1a_1b"]["fp"], res["1a_1b"]["fn"]
        p = tp/(tp+fp) if tp+fp > 0 else 0
        r = tp/(tp+fn) if tp+fn > 0 else 0
        f1 = 2*p*r/(p+r) if p+r > 0 else 0
        print(f"{'a+b: Entity Label':<25} | {p:.4f}  | {r:.4f}  | {f1:.4f}  | {'-':<8}")

    if "1c" in res:
        acc_c = res["1c"]["correct"]/res["1c"]["total"] if res["1c"]["total"] > 0 else 0
        print(f"{'c: Entity Attributes':<25} | {'-':<8} | {'-':<8} | {'-':<8} | {acc_c:.4f}")

    if "1d" in res:
        tp_d, fp_d, fn_d = res["1d"]["tp"], res["1d"]["fp"], res["1d"]["fn"]
        p_d = tp_d/(tp_d+fp_d) if tp_d+fp_d > 0 else 0
        r_d = tp_d/(tp_d+fn_d) if tp_d+fn_d > 0 else 0
        f1_d = 2*p_d*r_d/(p_d+r_d) if p_d+r_d > 0 else 0
        print(f"{'d: Relation Type':<25} | {p_d:.4f}  | {r_d:.4f}  | {f1_d:.4f}  | {'-':<8}")

    if "1e" in res:
        acc_e = res["1e"]["correct"]/res["1e"]["total"] if res["1e"]["total"] > 0 else 0
        print(f"{'e: Relation Attributes':<25} | {'-':<8} | {'-':<8} | {'-':<8} | {acc_e:.4f}")


# CONFIG 1 - Few-Shot (Task a, b, c, d, e)
Đầu vào: chỉ có text thô, `<TAGS/>` rỗng. Prompt kèm ví dụ minh hoạ từ training data.

In [12]:
# Đường dẫn dữ liệu Config 1
TEST_C1_DIR = "/kaggle/input/datasets/nguyenng11/spacenewdata/newdata/test/test.config1"
OUTPUT_C1 = "/kaggle/working/results_config1.jsonl"

# ========================================================
# FEW-SHOT CONFIG 1: Tìm entities + relations từ đầu
# ========================================================

# Prompt Giai đoạn 1: Tìm thực thể (có few-shot examples)
PROMPT_ENTITIES_C1 = """Extract all spatial entities from the text below.

Entity types:
- PLACE: Any geographic location, building, room, or landform.
- PATH: Any route, road, river, bridge, or trajectory.
- SPATIAL_ENTITY: Any spatial object not categorized as PLACE or PATH (e.g., people, vehicles, objects).
- MOTION: Any verb or phrase indicating movement (e.g., flew, walked, drove).
- NONMOTION_EVENT: Events that are not movement but have spatial relevance (e.g., stayed, lived).
- SPATIAL_SIGNAL: Prepositions or phrases indicating spatial relation (e.g., in, on, near, beside).
- MOTION_SIGNAL: Words indicating direction or manner of movement (e.g., from, to, through, north).

For each entity, provide: id (e0, e1, ...), text, label, and attributes.
Attributes depend on entity type:
- PLACE/PATH/SPATIAL_ENTITY: form (NAM/NOM), dimensionality (POINT/LINE/AREA/VOLUME), countable (TRUE/FALSE), dcl (TRUE/FALSE).
- MOTION: motion_class (MOVE/MOVE_INTERNAL/MOVE_EXTERNAL/LEAVE/REACH/CROSS/DEVIATE/FOLLOW/HIT), motion_sense (LITERAL/FICTIVE), motion_type (PATH/MANNER/COMPOUND).
- SPATIAL_SIGNAL: semantic_type (TOPOLOGICAL/DIRECTIONAL/DIR_TOP).
- MOTION_SIGNAL: motion_signal_type (PATH/MANNER).

Here are some examples of correct extractions:

{fewshot_examples}

Now extract entities from the following text. Return ONLY valid JSON:
{{"entities": [{{"id": "e0", "text": "...", "label": "PLACE", "attributes": {{"form": "NOM", "dimensionality": "AREA", "countable": "TRUE", "dcl": "FALSE"}}}}]}}

TEXT: {text_input}
"""

# Prompt Giai đoạn 2: Tìm quan hệ (có few-shot examples)
PROMPT_RELATIONS_C1 = """Given the text and entities below, extract all spatial relations.

Relation types:
- QSLink: Topological relation. relType can be IN, EC, DC, PO, EQ, NTPP, TPP.
  Attributes: trajector, landmark, trigger, relType.
- OLink: Orientation relation. frame_type can be ABSOLUTE, INTRINSIC, RELATIVE.
  Attributes: trajector, landmark, trigger, relType, frame_type, referencePt, projective.
- MoveLink: Movement relation connecting MOTION trigger to participants.
  Attributes: trigger, mover, source, goal, midPoint, landmark, pathID, motion_signalID, goal_reached.

Use ONLY the provided entity IDs.

Here are some examples of correct relation extractions:

{fewshot_examples}

Now extract relations from the following. Return ONLY valid JSON:
{{"relations": [{{"type": "QSLink", "trajector": "e0", "landmark": "e1", "trigger": "e2", "relType": "IN"}}, {{"type": "MoveLink", "trigger": "e3", "mover": "e0", "source": "e1", "goal": "e4", "goal_reached": "YES"}}]}}

TEXT: {text_input}
ENTITIES: {entities_json}
"""


In [13]:
def run_c1_inference(file_path):
    # Đọc văn bản từ file
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'xml')
        raw_text = soup.find('TEXT').get_text()[:MAX_TEXT_LENGTH]

    # BƯỚC 1: Tìm thực thể (Few-Shot)
    p1 = PROMPT_ENTITIES_C1.replace("{text_input}", raw_text).replace("{fewshot_examples}", ENTITY_EXAMPLES_STR)
    res1 = ask_gemma_raw(p1)
    entities_data = extract_json_only(res1).get("entities", [])

    # BƯỚC 2: Tìm quan hệ (Few-Shot)
    entities_str = json.dumps(entities_data, ensure_ascii=False)
    p2 = PROMPT_RELATIONS_C1.replace("{text_input}", raw_text).replace("{entities_json}", entities_str).replace("{fewshot_examples}", RELATION_EXAMPLES_STR)
    res2 = ask_gemma_raw(p2)
    relations_data = extract_json_only(res2).get("relations", [])

    return {
        "entities": entities_data,
        "relations": relations_data
    }

In [14]:
# Đọc danh sách file đã chạy (resume)
def get_done_files(output_path):
    done = set()
    if os.path.exists(output_path):
        with open(output_path, 'r') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done.add(r['metadata']['file'])
                except: pass
    return done

# Chạy inference Config 1
all_c1_files = glob.glob(os.path.join(TEST_C1_DIR, '**', '*.xml'), recursive=True)
done_c1 = get_done_files(OUTPUT_C1)
print(f"🚀 Config 1: {len(all_c1_files)} files, đã xong {len(done_c1)}, còn {len(all_c1_files)-len(done_c1)}")

for i, path in enumerate(all_c1_files):
    name = os.path.basename(path)
    if name in done_c1:
        print(f"[{i+1}/{len(all_c1_files)}] ⏭️ Skip (đã có): {name}")
        continue
    print(f"[{i+1}/{len(all_c1_files)}] Processing: {name}")

    try:
        final_data = run_c1_inference(path)
        record = {
            "metadata": {"file": name, "config": "C1"},
            "data": final_data
        }
        with open(OUTPUT_C1, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    except Exception as e:
        print(f"⚠️ Lỗi tại file {name}: {e}")

    clean_memory()

print(f"🏁 Xong Config 1! Kết quả tại: {OUTPUT_C1}")

🚀 Config 1: 16 files, đã xong 0, còn 16
[1/16] Processing: Mazatlan.xml


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[2/16] Processing: IntoTheAndes.xml
[3/16] Processing: Huaraz.xml
[4/16] Processing: MexicoCity.xml
[5/16] Processing: Guatemala.xml
[6/16] Processing: Amazon.xml
[7/16] Processing: Paramo.xml
[8/16] Processing: Manaus.xml
[9/16] Processing: LosAngeles.xml
[10/16] Processing: Colon.xml
[11/16] Processing: Hurricanes.xml
[12/16] Processing: 46_N_24_E.xml
[13/16] Processing: 46_N_28_E.xml
[14/16] Processing: 46_N_26_E.xml
[15/16] Processing: 47_N_26_E.xml
[16/16] Processing: 45_N_22_E.xml
🏁 Xong Config 1! Kết quả tại: /kaggle/working/results_config1.jsonl


### Evaluation Config 1

In [15]:
# Đánh giá Config 1
metrics_c1 = evaluate_config1(OUTPUT_C1, GOLD_DICT)
display_report(metrics_c1, "CONFIG 1")


===== KẾT QUẢ ĐÁNH GIÁ CONFIG 1 (FEW-SHOT) =====
BÀI TOÁN                  | P        | R        | F1       | ACC     
----------------------------------------------------------------------
a+b: Entity Label         | 0.7200  | 0.0123  | 0.0243  | -       
c: Entity Attributes      | -        | -        | -        | 0.9688
d: Relation Type          | 1.0000  | 0.0301  | 0.0584  | -       
e: Relation Attributes    | -        | -        | -        | 0.3263


# CONFIG 2 - Few-Shot (Task c, d, e)
Đầu vào: entities đã cho sẵn (id, start, end, text, label). Cần predict attributes + relations.

In [16]:
# # Đường dẫn dữ liệu Config 2
# TEST_C2_DIR = "/kaggle/input/datasets/nguyenng11/spacenewdata/newdata/test/test.config2"
# OUTPUT_C2 = "/kaggle/working/results_config2.jsonl"

# # ========================================================
# # FEW-SHOT CONFIG 2: Predict attributes + relations
# # Entities đã cho sẵn từ XML
# # ========================================================

# # Prompt Task c: Dự đoán attributes cho entities (có few-shot examples)
# PROMPT_ATTRIBUTES_C2 = """Given the text and the list of spatial entities below, predict the attributes for each entity.

# Attribute rules by entity type:
# - PLACE: form (NAM for proper names like 'Paris', NOM for common nouns like 'city'), dimensionality (POINT/LINE/AREA/VOLUME), countable (TRUE/FALSE), dcl (TRUE/FALSE), ctv (TRUE for City/Town/Village, otherwise empty).
# - PATH: form (NAM/NOM), dimensionality (POINT/LINE/AREA/VOLUME), countable (TRUE/FALSE), dcl (TRUE/FALSE).
# - SPATIAL_ENTITY: form (NAM/NOM), dimensionality (POINT/LINE/AREA/VOLUME), countable (TRUE/FALSE), dcl (TRUE/FALSE).
# - MOTION: motion_class (MOVE/MOVE_INTERNAL/MOVE_EXTERNAL/LEAVE/REACH/CROSS/DEVIATE/FOLLOW/HIT), motion_sense (LITERAL/FICTIVE), motion_type (PATH/MANNER/COMPOUND).
# - NONMOTION_EVENT: no special spatial attributes.
# - SPATIAL_SIGNAL: semantic_type (TOPOLOGICAL/DIRECTIONAL/DIR_TOP).
# - MOTION_SIGNAL: motion_signal_type (PATH/MANNER).

# Here are some examples of correct attribute predictions:

# {fewshot_examples}

# Now predict attributes for the following entities. Return ONLY valid JSON with the same entity IDs and their predicted attributes:
# {{"entities": [{{"id": "pl0", "attributes": {{"form": "NAM", "dimensionality": "VOLUME", "countable": "TRUE", "dcl": "FALSE"}}}}]}}

# TEXT: {text_input}
# ENTITIES: {entities_json}
# """

# # Prompt Task d+e: Dự đoán relations (có few-shot examples)
# PROMPT_RELATIONS_C2 = """Given the text and spatial entities below, extract all spatial relations.

# Relation types:
# - QSLink: Topological relation. relType can be IN, EC, DC, PO, EQ, NTPP, TPP.
#   Attributes: trajector (entity ID), landmark (entity ID), trigger (entity ID or empty), relType.
# - OLink: Orientation relation.
#   Attributes: trajector, landmark, trigger, relType, frame_type (ABSOLUTE/INTRINSIC/RELATIVE), referencePt, projective (TRUE/FALSE).
# - MoveLink: Movement relation.
#   Attributes: trigger (MOTION entity ID), mover (entity ID), source (entity ID), goal (entity ID), midPoint, landmark, pathID, motion_signalID, goal_reached (YES/NO/UNCERTAIN).

# Use ONLY the provided entity IDs.

# Here are some examples of correct relation extractions:

# {fewshot_examples}

# Now extract relations from the following. Return ONLY valid JSON:
# {{"relations": [{{"type": "QSLink", "trajector": "pl0", "landmark": "pl1", "trigger": "s0", "relType": "IN"}}]}}

# TEXT: {text_input}
# ENTITIES: {entities_json}
# """


In [17]:
# def parse_c2_entities(soup):
#     """Parse entities từ XML Config 2 (chỉ id, text, label - bỏ MEASURE)"""
#     tags = soup.find('TAGS')
#     if not tags:
#         return []

#     entities = []
#     for tag in tags.find_all(recursive=False):
#         label = tag.name.upper()
#         if label not in ENTITY_LABELS:
#             continue
#         if tag.get('start') == '-1':
#             continue

#         entities.append({
#             'id': tag.get('id', ''),
#             'text': tag.get('text', ''),
#             'label': label,
#             'start': tag.get('start', ''),
#             'end': tag.get('end', '')
#         })
#     return entities


# def run_c2_inference(file_path):
#     with open(file_path, 'r', encoding='utf-8') as f:
#         soup = BeautifulSoup(f.read(), 'xml')
#         raw_text = soup.find('TEXT').get_text()[:MAX_TEXT_LENGTH]

#     # Parse entities có sẵn từ XML
#     given_entities = parse_c2_entities(soup)
#     entities_str = json.dumps(given_entities, ensure_ascii=False)

#     # BƯỚC 1: Predict attributes cho entities (Task c) - Few-Shot
#     p1 = PROMPT_ATTRIBUTES_C2.replace("{text_input}", raw_text).replace("{entities_json}", entities_str).replace("{fewshot_examples}", ATTRIBUTE_EXAMPLES_STR)
#     res1 = ask_gemma_raw(p1)
#     predicted_attrs = extract_json_only(res1).get("entities", [])

#     # Merge attributes vào entities gốc
#     attrs_map = {e.get('id'): e.get('attributes', {}) for e in predicted_attrs}
#     for ent in given_entities:
#         ent['attributes'] = attrs_map.get(ent['id'], {})

#     # BƯỚC 2: Predict relations (Task d, e) - Few-Shot
#     p2 = PROMPT_RELATIONS_C2.replace("{text_input}", raw_text).replace("{entities_json}", entities_str).replace("{fewshot_examples}", RELATION_EXAMPLES_STR)
#     res2 = ask_gemma_raw(p2)
#     relations_data = extract_json_only(res2).get("relations", [])

#     return {
#         "entities": given_entities,
#         "relations": relations_data
#     }

In [18]:
# # Chạy inference Config 2
# all_c2_files = glob.glob(os.path.join(TEST_C2_DIR, '**', '*.xml'), recursive=True)
# done_c2 = get_done_files(OUTPUT_C2)
# print(f"🚀 Config 2: {len(all_c2_files)} files, đã xong {len(done_c2)}, còn {len(all_c2_files)-len(done_c2)}")

# for i, path in enumerate(all_c2_files):
#     name = os.path.basename(path)
#     if name in done_c2:
#         print(f"[{i+1}/{len(all_c2_files)}] ⏭️ Skip: {name}")
#         continue
#     print(f"[{i+1}/{len(all_c2_files)}] Processing: {name}")

#     try:
#         final_data = run_c2_inference(path)
#         record = {
#             "metadata": {"file": name, "config": "C2"},
#             "data": final_data
#         }
#         with open(OUTPUT_C2, 'a', encoding='utf-8') as f:
#             f.write(json.dumps(record, ensure_ascii=False) + '\n')
#     except Exception as e:
#         print(f"⚠️ Lỗi tại file {name}: {e}")

#     clean_memory()

# print(f"🏁 Xong Config 2! Kết quả tại: {OUTPUT_C2}")

### Evaluation Config 2 (Task c, d, e)

In [19]:
# # Đánh giá Config 2
# metrics_c2 = evaluate_config2(OUTPUT_C2, GOLD_DICT)
# display_report(metrics_c2, "CONFIG 2")

# CONFIG 3 - Few-Shot (Task d, e)
Đầu vào: entities + đầy đủ attributes. Chỉ cần predict relations.

In [20]:
# # Đường dẫn dữ liệu Config 3
# TEST_C3_DIR = "/kaggle/input/datasets/nguyenng11/spacenewdata/newdata/test/test.config3"
# OUTPUT_C3 = "/kaggle/working/results_config3.jsonl"

# # ========================================================
# # FEW-SHOT CONFIG 3: Chỉ predict relations
# # Entities + attributes đã cho sẵn
# # ========================================================

# # Prompt Task d+e: Dự đoán relations (có thêm thông tin attributes + few-shot examples)
# PROMPT_RELATIONS_C3 = """Given the text and spatial entities with their attributes below, extract all spatial relations.

# Relation types:
# - QSLink: Topological relation between spatial entities.
#   Attributes: trajector (entity ID), landmark (entity ID), trigger (signal entity ID or empty), relType (IN/EC/DC/PO/EQ/NTPP/TPP).
# - OLink: Orientation relation.
#   Attributes: trajector, landmark, trigger, relType (direction like ABOVE/BELOW/BESIDE/TOWARD/SOUTHWEST etc.), frame_type (ABSOLUTE/INTRINSIC/RELATIVE), referencePt, projective (TRUE/FALSE).
# - MoveLink: Movement relation connecting a MOTION trigger to spatial participants.
#   Attributes: trigger (MOTION entity ID), mover (entity ID), source (entity ID or empty), goal (entity ID or empty), midPoint (entity ID or empty), landmark (entity ID or empty), pathID (PATH entity ID or empty), motion_signalID (MOTION_SIGNAL ID(s) or empty), goal_reached (YES/NO/UNCERTAIN or empty).

# Important:
# - Use the entity attributes (like motion_class, dimensionality) to help determine the correct relations.
# - MOTION entities are triggers for MoveLink.
# - SPATIAL_SIGNAL entities are triggers for QSLink/OLink.
# - Use ONLY the provided entity IDs.

# Here are some examples of correct relation extractions:

# {fewshot_examples}

# Now extract relations from the following. Return ONLY valid JSON:
# {{"relations": [{{"type": "QSLink", "trajector": "pl0", "landmark": "pl1", "trigger": "s0", "relType": "IN"}}, {{"type": "MoveLink", "trigger": "m0", "mover": "se0", "source": "pl0", "goal": "pl1", "midPoint": "", "landmark": "", "pathID": "p0", "motion_signalID": "ms0", "goal_reached": "YES"}}]}}

# TEXT: {text_input}
# ENTITIES: {entities_json}
# """


In [21]:
# def parse_c3_entities(soup):
#     """Parse entities từ XML Config 3 (id, text, label + tất cả attributes - bỏ MEASURE)"""
#     tags = soup.find('TAGS')
#     if not tags:
#         return []

#     entities = []
#     for tag in tags.find_all(recursive=False):
#         label = tag.name.upper()
#         if label not in ENTITY_LABELS:
#             continue
#         if tag.get('start') == '-1':
#             continue

#         # Lấy tất cả attributes
#         attrs = {}
#         for attr, val in tag.attrs.items():
#             if attr in ['id', 'start', 'end', 'text', 'comment']:
#                 continue
#             attrs[attr] = val

#         entities.append({
#             'id': tag.get('id', ''),
#             'text': tag.get('text', ''),
#             'label': label,
#             'attributes': attrs
#         })
#     return entities


# def run_c3_inference(file_path):
#     with open(file_path, 'r', encoding='utf-8') as f:
#         soup = BeautifulSoup(f.read(), 'xml')
#         raw_text = soup.find('TEXT').get_text()[:MAX_TEXT_LENGTH]

#     # Parse entities có sẵn từ XML (bao gồm attributes)
#     given_entities = parse_c3_entities(soup)

#     # Chia entities thành batches nhỏ để tránh OOM
#     BATCH_SIZE = 30
#     all_relations = []

#     for start in range(0, len(given_entities), BATCH_SIZE):
#         batch = given_entities[start:start+BATCH_SIZE]
#         batch_str = json.dumps(batch, ensure_ascii=False)
#         p1 = PROMPT_RELATIONS_C3.replace("{text_input}", raw_text).replace("{entities_json}", batch_str).replace("{fewshot_examples}", RELATION_EXAMPLES_STR)
#         res1 = ask_gemma_raw(p1)
#         rels = extract_json_only(res1).get("relations", [])
#         all_relations.extend(rels)

#     return {
#         "entities": given_entities,
#         "relations": all_relations
#     }

In [22]:
# # Chạy inference Config 3
# all_c3_files = glob.glob(os.path.join(TEST_C3_DIR, '**', '*.xml'), recursive=True)
# done_c3 = get_done_files(OUTPUT_C3)
# print(f"🚀 Config 3: {len(all_c3_files)} files, đã xong {len(done_c3)}, còn {len(all_c3_files)-len(done_c3)}")

# for i, path in enumerate(all_c3_files):
#     name = os.path.basename(path)
#     if name in done_c3:
#         print(f"[{i+1}/{len(all_c3_files)}] ⏭️ Skip: {name}")
#         continue
#     print(f"[{i+1}/{len(all_c3_files)}] Processing: {name}")

#     try:
#         final_data = run_c3_inference(path)
#         record = {
#             "metadata": {"file": name, "config": "C3"},
#             "data": final_data
#         }
#         with open(OUTPUT_C3, 'a', encoding='utf-8') as f:
#             f.write(json.dumps(record, ensure_ascii=False) + '\n')
#     except Exception as e:
#         print(f"⚠️ Lỗi tại file {name}: {e}")

#     clean_memory()

# print(f"🏁 Xong Config 3! Kết quả tại: {OUTPUT_C3}")

### Evaluation Config 3 (Task d, e)

In [23]:
# # Đánh giá Config 3
# metrics_c3 = evaluate_config3(OUTPUT_C3, GOLD_DICT)
# display_report(metrics_c3, "CONFIG 3")

# Tổng kết kết quả 3 Config

In [24]:
# print("=" * 70)
# print("  TỔNG KẾT KẾT QUẢ FEW-SHOT SPACEEVAL")
# print("=" * 70)
# display_report(metrics_c1, "CONFIG 1 (a,b,c,d,e)")
# display_report(metrics_c2, "CONFIG 2 (c,d,e)")
# display_report(metrics_c3, "CONFIG 3 (d,e)")
